# InvoiceAI — REST API (FastAPI)

Runs the API in Colab, tests it, and gives you a link to the automatic `/docs` page.

**Before you start:** `Runtime → Change runtime type → T4 GPU`.

## 1. Check the GPU

In [ ]:
!nvidia-smi

## 2. Get the code

In [ ]:
GITHUB_USER = "YOUR_GITHUB_USERNAME"  # <- change this
REPO_DIR = "/content/invoice-ai"

import os
if os.path.exists(REPO_DIR):
    !git -C {REPO_DIR} pull
else:
    !git clone https://github.com/{GITHUB_USER}/invoice-ai.git {REPO_DIR}
!ls {REPO_DIR}/api {REPO_DIR}/tests

## 3. Install requirements

If Colab asks you to **restart the session**, click Restart and continue with step 4.

In [ ]:
!pip install -q -r /content/invoice-ai/requirements.txt

## 4. Run the tests

These use a fake model, so they take a few seconds and need no GPU. All should pass.

In [ ]:
!cd /content/invoice-ai && python -m pytest

## 5. Start the API

The API runs in the background. Loading the model takes 1–3 minutes (first time: download ~7 GB).

In [ ]:
import subprocess, time, requests

REPO_DIR = "/content/invoice-ai"
API = "http://localhost:8000"
log_file = open("/content/api.log", "w")
api_process = subprocess.Popen(
    ["python", "-m", "uvicorn", "api.main:app", "--host", "0.0.0.0", "--port", "8000"],
    cwd=REPO_DIR, stdout=log_file, stderr=subprocess.STDOUT,
)

start = time.time()
while True:
    if api_process.poll() is not None:
        print("The API stopped. Last lines of the log:")
        print(open("/content/api.log").read()[-3000:])
        break
    try:
        health = requests.get(f"{API}/health", timeout=5)
        if health.status_code == 200:
            print(f"API ready after {time.time() - start:.0f} s:", health.json())
            break
        if health.json().get("status") == "error":
            print("Model loading failed:", health.json())
            break
    except requests.exceptions.ConnectionError:
        pass
    if time.time() - start > 900:
        print("Still not ready after 15 minutes. Check /content/api.log")
        break
    time.sleep(5)

## 6. Test `/extract` with one sample file

In [ ]:
import json
from pathlib import Path

samples = sorted(p for p in Path(f"{REPO_DIR}/samples").iterdir() if p.suffix.lower() in {".jpg", ".jpeg", ".png", ".pdf"})
print("Samples:", [p.name for p in samples])

with open(samples[0], "rb") as f:
    response = requests.post(f"{API}/extract", files={"file": (samples[0].name, f)}, timeout=300)
print("Status code:", response.status_code)
body = response.json()
print(json.dumps(body["invoice"], indent=2, ensure_ascii=False))
print("\nConfidence:", {name: field["confidence"] for name, field in body["fields"].items()})
print("Warnings:", body["warnings"])
print("Time:", body["processing_seconds"], "s")

## 7. Test error messages

A wrong file type should give a clear error, not a crash.

In [ ]:
bad = requests.post(f"{API}/extract", files={"file": ("notes.txt", b"hello")})
print(bad.status_code, bad.json())
fake = requests.post(f"{API}/extract", files={"file": ("fake.pdf", b"this is not a pdf")})
print(fake.status_code, fake.json())

## 8. Batch + Excel download

We send all samples plus one PDF made from the first sample. Then we open the Excel file to check the sheets.

In [ ]:
import pandas as pd
from PIL import Image
from IPython.display import display

pdf_path = Path("/content/sample_invoice.pdf")
Image.open(samples[0]).convert("RGB").save(pdf_path)
upload = [("files", (p.name, open(p, "rb"))) for p in samples[:3]] + [("files", (pdf_path.name, open(pdf_path, "rb")))]

response = requests.post(f"{API}/export/excel", files=upload, timeout=900)
print("Status code:", response.status_code)
excel_path = "/content/invoices.xlsx"
with open(excel_path, "wb") as f:
    f.write(response.content)

sheets = pd.read_excel(excel_path, sheet_name=None)
print("Sheets:", list(sheets))
for name, frame in sheets.items():
    print(f"\n--- {name} ---")
    display(frame)

In [ ]:
from google.colab import files
files.download(excel_path)  # open it in Excel: yellow/red cells = please check

## 9. Open the `/docs` page (only you)

This link works in **your** browser while this notebook runs (it uses your Google login).
On the `/docs` page, click an endpoint → **Try it out** → choose a file → **Execute**.

In [ ]:
from google.colab.output import eval_js
print(eval_js("google.colab.kernel.proxyPort(8000)") + "docs")

## 10. Public link (optional)

A free Cloudflare tunnel gives a public URL, so you can test from your laptop with `curl` or show it to someone.
⚠️ **Anyone with the link can use your API.** Stop it (step 11) when you are done.

In [ ]:
import re, threading

!wget -q -O /content/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x /content/cloudflared

tunnel = subprocess.Popen(["/content/cloudflared", "tunnel", "--url", "http://localhost:8000", "--no-autoupdate"],
                          stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
PUBLIC_URL = None
start = time.time()
while PUBLIC_URL is None and time.time() - start < 60:
    line = tunnel.stdout.readline()
    match = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line)
    if match:
        PUBLIC_URL = match.group(0)

# keep reading the tunnel output in the background, so it never blocks
threading.Thread(target=lambda: [None for _ in tunnel.stdout], daemon=True).start()

if PUBLIC_URL:
    print("Public API: ", PUBLIC_URL)
    print("Docs page:  ", PUBLIC_URL + "/docs")
    print("\nTest from your laptop:")
    print(f'  curl {PUBLIC_URL}/health')
    print(f'  curl -F "file=@receipt.jpg" {PUBLIC_URL}/extract')
    print(f'  curl -F "files=@a.jpg" -F "files=@b.pdf" {PUBLIC_URL}/export/excel -o invoices.xlsx')
else:
    print("No tunnel URL found. Use the private link from step 9 instead.")

## 11. Stop everything

In [ ]:
for name, process in [("tunnel", globals().get("tunnel")), ("API", globals().get("api_process"))]:
    if process is not None and process.poll() is None:
        process.terminate()
        print(f"Stopped the {name}.")